# 🧪 Mutual Fund Facts RAG — Repeatable Evaluation Pipeline

This Google Colab notebook runs a repeatable evaluation suite for your mutual fund RAG chatbot before deploying changes to production.

### Capabilities
1. **Fixed Evaluation Dataset**: Reads and validates `test_dataset.json` (30 curated questions covering factual, multi-hop, procedural, and unanswerable/guardrail scenarios).
2. **Real Chatbot Execution**: Executes queries against either the **in-process RAG pipeline** or a **live HTTP REST endpoint**.
3. **DeepEval Metric Suite**:
   - **Answer Relevancy**: Assesses whether the response directly addresses the question.
   - **Faithfulness**: Measures whether generated statements are grounded in retrieved evidence.
   - **Contextual Recall**: Evaluates whether retrieved chunks cover all ground-truth facts.
4. **Groq as LLM-as-a-Judge**: Custom `GroqJudge` leveraging `qwen/qwen3.8-27b` or `llama-3.3-70b-versatile` with reasoning tag sanitization and rate-limit backoff.
5. **Unanswerable / Refusal Handling**: Distinguishes unavailable metrics (`N/A`) from `0.0` to avoid artificial score skewing.
6. **Performance & Cost Tracking**: Measures retrieval, generation, and total end-to-end latency; captures token usage and computes exact costs using published pricing.
7. **Complete Output Artifacts**: Generates `evaluation_results.csv`, `evaluation_summary.json`, and `evaluation_visualizations.png`.

---  
## 📦 Step 1: Environment Setup & Dependencies
Install the required packages in this Colab session.

In [ ]:
# Install evaluation dependencies
!pip install -q deepeval groq chromadb sentence-transformers matplotlib pandas pydantic python-dotenv requests

---  
## 📂 Step 2: Codebase & Vectorstore Availability
If running in Colab, choose ONE of the options below to make your repository files and ChromaDB index available:
- **Option A (GitHub Clone)**: Clone your repository if hosted on GitHub.
- **Option B (Google Drive Mount)**: If your project is stored in Google Drive.
- **Option C (Direct Upload)**: Upload your project folder or `test_dataset.json`.

In [ ]:
import os
from pathlib import Path

# Check if running inside Google Colab
IN_COLAB = "google.colab" in sys.modules if "sys" in locals() else False

# If using GitHub clone:
# !git clone https://github.com/<YOUR_USER>/mf-facts-rag.git
# %cd mf-facts-rag

# Verify current directory has test_dataset.json
dataset_file = Path("test_dataset.json")
print(f"Current working directory: {Path.cwd()}")
print(f"Dataset exists: {dataset_file.exists()}")

---  
## 🔑 Step 3: Secure API Key Configuration
Set your `GROQ_API_KEY` using **Google Colab Secrets** (recommended) or environment variables.  
Never print or hardcode secrets in code cells!

In [ ]:
import os

try:
    from google.colab import userdata
    # Reads from Colab Secrets (Left sidebar -> Key icon)
    groq_key = userdata.get("GROQ_API_KEY")
    if groq_key:
        os.environ["GROQ_API_KEY"] = str(groq_key).strip()
    openai_key = userdata.get("OPENAI_API_KEY")
    if openai_key:
        os.environ["OPENAI_API_KEY"] = str(openai_key).strip()
except Exception:
    pass

# Verification (without revealing the key)
if os.getenv("GROQ_API_KEY"):
    print("✓ GROQ_API_KEY is configured successfully.")
else:
    print("⚠️ GROQ_API_KEY is missing! Set it in Colab Secrets or enter it below:")
    import getpass
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ").strip()

---  
## ⚙️ Step 4: Pipeline Configuration Block
All evaluation settings are grouped in this single configuration cell.

In [ ]:
# ==============================================================================
# ⚙️ EVALUATION CONFIGURATION
# ==============================================================================

# 1. Pipeline Execution Mode:
#    - "direct_pipeline": Executes your local Python RAG pipeline in-process
#    - "http_api": Calls an external live HTTP REST endpoint
EVALUATION_MODE = "direct_pipeline"

# 2. Live HTTP API Settings (Used ONLY if EVALUATION_MODE == "http_api")
API_ENDPOINT_URL = "https://your-live-deployment-api.com/api/chat"
API_AUTH_TOKEN = ""

# 3. Chatbot Model Configuration
GROQ_MODEL = "qwen/qwen3.8-27b"
os.environ["GROQ_MODEL"] = GROQ_MODEL

# 4. DeepEval LLM-as-a-Judge Configuration
#    Options: "groq" or "openai"
JUDGE_PROVIDER = "groq"
JUDGE_MODEL = "qwen/qwen3.8-27b"  # or "llama-3.3-70b-versatile"
JUDGE_DELAY_SECONDS = 0.5         # Inter-request delay to respect Groq rate limits

# 5. Dataset & Output Paths
DATASET_PATH = "test_dataset.json"
OUTPUT_DIR = "evaluation/outputs"

# 6. Execution Limit (Set to None to evaluate all 30 questions, or e.g. 5 for a quick test)
SAMPLE_LIMIT = None
# ==============================================================================

---  
## 🚀 Step 5: Run Evaluation Pipeline
Execute the evaluation script with your configuration.

In [ ]:
from evaluation.run_eval import run_evaluation

run_evaluation(
    mode=EVALUATION_MODE,
    dataset_path=DATASET_PATH,
    output_dir=OUTPUT_DIR,
    judge_provider=JUDGE_PROVIDER,
    judge_model=JUDGE_MODEL,
    groq_model=GROQ_MODEL,
    api_url=API_ENDPOINT_URL,
    limit=SAMPLE_LIMIT,
    judge_delay=JUDGE_DELAY_SECONDS,
)

---  
## 📊 Step 6: Inspect Results & Visualizations
Review per-question scores, aggregate metrics, and diagnostic plots.

In [ ]:
import pandas as pd
from IPython.display import Image, display

# 1. Display per-sample CSV table
csv_path = f"{OUTPUT_DIR}/evaluation_results.csv"
df = pd.read_csv(csv_path)
print(f"✓ Loaded {len(df)} evaluation records:")
display(df[["id", "category", "question", "answer_relevancy_score", "faithfulness_score", "contextual_recall_score", "total_latency_ms", "status"]].head(10))

# 2. Display Visualization Charts
plot_path = f"{OUTPUT_DIR}/evaluation_visualizations.png"
if Path(plot_path).exists():
    display(Image(filename=plot_path))
else:
    print("Visualization plot not found.")

---  
## 💾 Step 7: Download Outputs from Google Colab
Download your CSV report, JSON summary, and PNG visualizations to your local machine.

In [ ]:
try:
    from google.colab import files
    import zipfile

    zip_name = "rag_evaluation_outputs.zip"
    with zipfile.ZipFile(zip_name, "w") as zipf:
        for fname in ["evaluation_results.csv", "evaluation_summary.json", "evaluation_visualizations.png"]:
            fpath = Path(OUTPUT_DIR) / fname
            if fpath.exists():
                zipf.write(fpath, arcname=fname)
                print(f"Added {fname} to {zip_name}")

    print(f"Downloading {zip_name}...")
    files.download(zip_name)
except Exception as exc:
    print(f"Note: Running outside Colab or download skipped ({exc}). Files are available in {OUTPUT_DIR}.")